### File used to merge outputs from NLP models on structured data.

In [28]:
import pandas as pd

1. Merge toxic messages in political messages

In [29]:
flat_political = pd.read_csv('../clean_data/flat_political_interactions.csv')

flat_toxic = pd.read_csv('../clean_data/flat_toxic_interactions.csv', index_col=0)
flat_toxic = flat_toxic.sort_values("toxicity_level", ascending=False).drop_duplicates(subset="text", keep="first")
flat_toxic['violent'] = (flat_toxic['toxicity_level'] >= flat_toxic['toxicity_level'].quantile(0.75)).astype(int)
flat_toxic = flat_toxic.drop(columns=['toxicity_level'])

In [30]:
df_merge = pd.merge(flat_political, flat_toxic, on='text', how='left', indicator=True)
print(df_merge['_merge'].value_counts())
# All messages in toxic are included in political 
df_merge = df_merge.drop(columns=['_merge'])

_merge
both          151137
left_only          0
right_only         0
Name: count, dtype: int64


2. Merge Left-Right in political messages

In [31]:
flat_left_right = pd.read_csv('../clean_data/flat_left_right_interactions.csv')
df_merge = pd.merge(df_merge, flat_left_right, on='text', how='left', indicator=True)
print(df_merge['_merge'].value_counts())
# All messages in left-right are included in political. 
df_merge = df_merge.drop(columns=['_merge'])

_merge
both          151137
left_only          0
right_only         0
Name: count, dtype: int64


3. Merge political messages in all messages 

In [32]:
flat_interactions = pd.read_csv('../clean_data/flat_all_interactions.csv')
df_merge = pd.merge(flat_interactions, df_merge, on='text', how='left', indicator='political')

In [33]:
print(df_merge['political'].value_counts())
# All political messages are in flat messages
df_merge['political'] = df_merge['political'].replace({
    'left_only': 0,
    'both':1
    }).cat.remove_unused_categories()
print(df_merge['political'].value_counts())

political
left_only     708515
both          151137
right_only         0
Name: count, dtype: int64
political
0    708515
1    151137
Name: count, dtype: int64


C:\Users\cfrou\AppData\Local\Temp\ipykernel_37504\613558674.py:3: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df_merge['political'] = df_merge['political'].replace({


4. Merge all flat messages on all structured interactions

In [34]:
struct_interactions = pd.read_csv('../clean_data/struc_all_interactions.csv')
struct_interactions = struct_interactions.rename(columns={
    'text':'user_text',
    'text_inter':'inter_text'
})
struct_interactions = struct_interactions.drop(columns=['pfp_inter', 'after', 'date_ins'])

In [35]:
struct_panel = pd.merge(struct_interactions, df_merge, left_on='user_text', right_on='text', how='left')
struct_panel = struct_panel.rename(columns={
    'violent':'user_violent', 
    'left_right':'user_left_right', 
    'political':'user_political'
    })
struct_panel = struct_panel.drop(columns=['text'])

In [36]:
struct_panel = pd.merge(struct_panel, df_merge, left_on='inter_text', right_on='text', how='left')
struct_panel = struct_panel.rename(columns={
    'violent':'inter_violent', 
    'left_right':'inter_left_right', 
    'political':'inter_political'
    })
struct_panel = struct_panel.drop(columns=['text'])

In [37]:
struct_panel = pd.merge(struct_panel, df_merge, left_on='post_text', right_on='text', how='left')
struct_panel = struct_panel.rename(columns={
    'violent':'post_violent', 
    'left_right':'post_left_right', 
    'political':'post_political'
    })
struct_panel = struct_panel.drop(columns=['text'])

In [38]:
struct_panel.columns

Index(['id', 'user', 'type', 'date', 'user_text', 'user_inter', 'inter_text',
       'post_title', 'post_text', 'language', 'user_violent',
       'user_left_right', 'user_political', 'inter_violent',
       'inter_left_right', 'inter_political', 'post_violent',
       'post_left_right', 'post_political'],
      dtype='object')

In [39]:
struct_panel.to_csv('../clean_data/struct_annotated_interactions.csv', index=False)